# Simple Fine-tuning Baseline

This notebook produces a simple fine-tuning-based baseline submission for the Neural Debris Removal in Streak Detection Models.

**Workflow**
1. Install Detectron2 + deps
2. Load the **poisoned** RetinaNet model
3. Fine-tune on the **unlearn set** with *empty* labels — the naive unlearning step
4. Run inference on the test set
5. Save `submission.csv`

## 1. Install Detectron2

In [1]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 5.5 MB/s eta 0:00:00


## 2. Imports

In [2]:
import copy
import json
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.data import (
    DatasetCatalog,
    DatasetMapper,
    MetadataCatalog,
    build_detection_train_loader,
    detection_utils as utils,
)
from detectron2.engine import DefaultPredictor, DefaultTrainer
from tqdm import tqdm

## 3. Configuration

The **model-architecture** block must match how the poisoned model was trained; changing those values will cause silent weight mismatches (new random detection heads) and your submission will score poorly.

In [3]:
# ── Input paths (Kaggle Datasets) ──
POISONED_WEIGHTS = "/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/poisoned_model/poisoned_model.pth"
UNLEARN_DIR       = "/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/unlearn_set"
TEST_DIR         = "/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/test_set/test_set"

# ── Output paths ──
OUTPUT_DIR      = "/kaggle/working/unlearned"
SUBMISSION_PATH = "/kaggle/working/submission.csv"

# ── Model architecture (MUST match the poisoned model's training config) ──
BASE_CONFIG          = "COCO-Detection/retinanet_R_50_FPN_3x.yaml"
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES         = [[16], [32], [64], [128], [256]]
NUM_CLASSES          = 1

# ── Unlearning hyperparameters (feel free to sweep) ──
UNLEARN_LR    = 1e-4
UNLEARN_ITERS = 20
BATCH_SIZE    = 4

# ── Inference ──
CONF_THRESH = 0.2
IMG_W = IMG_H = 1024

## 4. Register the unlearn set

The unlearn set consists of images whose annotations we **discard**. Training on these tells the model "there is nothing of interest in these images" — which suppresses the object detections the poisoned model learned.

In [4]:
UNLEARN_DATASET = "unlearn"

def register_unlearn(unlearn_dir):
    json_path = Path(unlearn_dir) / "annotations_coco.json"
    with open(json_path) as f:
        coco = json.load(f)
    dicts = [
        {
            "file_name": str(Path(unlearn_dir) / im["file_name"]),
            "height":    im["height"],
            "width":     im["width"],
            "image_id":  im["id"],
            "annotations": [],   # ← the 'unlearning' signal
        }
        for im in coco["images"]
    ]
    DatasetCatalog.register(UNLEARN_DATASET, lambda: dicts)
    MetadataCatalog.get(UNLEARN_DATASET).set(thing_classes=["object"])
    print(f"Registered unlearn set: {len(dicts)} images (empty annotations)")

register_unlearn(UNLEARN_DIR)

Registered unlearn set: 20 images (empty annotations)


## 5. 16-bit image handling

The input images are **uint16 single-channel PNGs**. To be compatible with the Detectron2 framework without losing the original numerical precision, we scale them to float32 in 0–255 range. We also override Detectron2's default `DatasetMapper` to:
- Read with `IMREAD_UNCHANGED`
- Duplicate the single channel to 3 channels (Detectron2 expects BGR-like 3-channel input)
- Skip the default filter that drops images with no annotations (we *need* them for unlearning).

In [5]:
class UInt16DatasetMapper(DatasetMapper):
    """Reads 16-bit PNGs as float32 in [0, 255] and attaches empty instances (the unlearning signal)."""
    def __call__(self, dataset_dict):
        dataset_dict = copy.deepcopy(dataset_dict)

        image = cv2.imread(dataset_dict["file_name"], cv2.IMREAD_UNCHANGED)
        if image.dtype == np.uint16:
            image = image.astype(np.float32) / 65535.0
        image = np.clip(image * 255, 0, 255).astype(np.float32)
        if image.ndim == 2:
            image = np.repeat(image[:, :, None], 3, axis=2)

        dataset_dict["image"] = torch.as_tensor(image.transpose(2, 0, 1).copy())
        dataset_dict["instances"] = utils.annotations_to_instances([], image.shape[:2])
        return dataset_dict


class UnlearnTrainer(DefaultTrainer):
    """Train loader that keeps images without annotations (empty-label training)."""
    @classmethod
    def build_train_loader(cls, cfg):
        dataset_dicts = DatasetCatalog.get(cfg.DATASETS.TRAIN[0])
        mapper = UInt16DatasetMapper(cfg, is_train=True, augmentations=[])
        return build_detection_train_loader(cfg, mapper=mapper, dataset=dataset_dicts)

## 6. Build config and run unlearning

Builds the Detectron2 `CfgNode` from the base RetinaNet recipe, overrides anchors + weights to match the poisoned model, then fine-tunes for a handful of iterations on the unlearn set.

In [6]:
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))

cfg.MODEL.WEIGHTS = POISONED_WEIGHTS
cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES

cfg.DATASETS.TRAIN = (UNLEARN_DATASET,)
cfg.DATASETS.TEST  = ()

cfg.DATALOADER.NUM_WORKERS = 2
cfg.SOLVER.IMS_PER_BATCH   = BATCH_SIZE
cfg.SOLVER.BASE_LR         = UNLEARN_LR
cfg.SOLVER.MAX_ITER        = UNLEARN_ITERS
cfg.SOLVER.STEPS           = []

cfg.OUTPUT_DIR = OUTPUT_DIR
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

trainer = UnlearnTrainer(cfg)
trainer.resume_or_load(resume=False)
trainer.train()

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[04/21 23:32:55 d2.engine.defaults]: Model:
RetinaNet(
  (backbone): FPN(
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelP6P7(
      (p6): Conv2d(2048, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (p7): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    )
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res2)

/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


[04/21 23:33:21 d2.utils.events]:  eta: 0:00:00  iter: 19  total_loss: 0.05361  loss_cls: 0.05361  loss_box_reg: 0    time: 0.9265  last_time: 0.8997  data_time: 0.0284  last_data_time: 0.0155   lr: 9.5005e-05  max_mem: 3162M


2026-04-21 23:33:25.667087: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776814406.106936      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776814406.203429      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776814407.152175      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776814407.152209      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776814407.152212      23 computation_placer.cc:177] computation placer alr

[04/21 23:33:55 d2.engine.hooks]: Overall training speed: 18 iterations in 0:00:16 (0.9265 s / it)
[04/21 23:33:55 d2.engine.hooks]: Total training time: 0:00:50 (0:00:34 on hooks)


## 7. Inference & submission

Iterate over test images, keep detections with `confidence >= CONF_THRESH`, and assemble each image's `prediction_string` as `"conf x y w h conf x y w h ..."` (space-delimited, see the submission format in the Overview tab).

In [7]:
cfg.MODEL.WEIGHTS = str(Path(OUTPUT_DIR) / "model_final.pth")
cfg.MODEL.RETINANET.SCORE_THRESH_TEST = CONF_THRESH
predictor = DefaultPredictor(cfg)


def load_for_inference(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype == np.uint16:
        im = im.astype(np.float32) / 65535.0
    im = np.clip(im * 255, 0, 255).astype(np.float32)
    if im.ndim == 2:
        im = np.repeat(im[:, :, None], 3, axis=2)
    return im


test_files = sorted(Path(TEST_DIR).glob("*.png"))
print(f"Running inference on {len(test_files)} images...")

rows = []
for img_path in tqdm(test_files, desc="Inference"):
    im = load_for_inference(img_path)
    out = predictor(im)["instances"].to("cpu")
    boxes  = out.pred_boxes.tensor.numpy()
    scores = out.scores.numpy()

    parts = []
    for (x1, y1, x2, y2), s in zip(boxes, scores):
        x1 = float(np.clip(x1, 0, IMG_W))
        y1 = float(np.clip(y1, 0, IMG_H))
        x2 = float(np.clip(x2, 0, IMG_W))
        y2 = float(np.clip(y2, 0, IMG_H))
        w, h = max(0.0, x2 - x1), max(0.0, y2 - y1)
        if w == 0 or h == 0:
            continue
        parts.extend([f"{float(s):.6f}", f"{x1:.2f}", f"{y1:.2f}", f"{w:.2f}", f"{h:.2f}"])

    # Use a single space for empty predictions so Kaggle's null-check passes.
    rows.append({"image_id": img_path.stem, "prediction_string": " ".join(parts) or " "})

submission = pd.DataFrame(rows)
submission.insert(0, "id", range(len(submission)))
submission.to_csv(SUBMISSION_PATH, index=False)
print(f"Wrote {SUBMISSION_PATH}  ({len(submission)} rows)")
submission.head()

[04/21 23:33:56 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/unlearned/model_final.pth ...
Running inference on 2000 images...


Inference: 100%|██████████| 2000/2000 [04:31<00:00,  7.36it/s]

Wrote /kaggle/working/submission.csv  (2000 rows)


,id,image_id,prediction_string
0,0,0,0.586042 894.05 186.30 9.02 40.99 0.490145 208...
1,1,1,
2,2,10,0.759720 5.05 17.87 37.02 58.52 0.285000 28.97...
3,3,100,0.307838 998.14 769.85 21.63 52.83
4,4,1000,
